# Job-Market Anxiety vs. Official Unemployment — Solution Notebook

Complete worked example that accompanies the Practice Skeleton.  
Uses carefully constructed synthetic labor-market series so the notebook is fully reproducible. Replace the synthetic block with real BLS unemployment and job-posting data for the final Capstone.

## 0. Setup

In [ ]:
library(gtrendsR)
library(dplyr)
library(ggplot2)
library(lubridate)
library(tidyr)
library(readr)

options(scipen = 999)
set.seed(42)

## 1. Acquire Google Trends Data
(Live call — results will vary by date. For reproducibility you can save the object once and reload it.)

In [ ]:
keywords <- c("unemployment benefits", "layoff", "job search", "remote jobs")

trends <- gtrends(keyword = keywords,
                  geo = "US",
                  time = "2015-01-01 2024-06-30",
                  onlyInterest = TRUE)

iot <- trends$interest_over_time
head(iot)
str(iot)

## 2. Clean Trends Data

In [ ]:
iot_clean <- iot %>%
  mutate(
    date = as.Date(date),
    hits = as.numeric(ifelse(hits == "<1", 0, hits)),
    year_month = floor_date(date, unit = "month")
  ) %>%
  group_by(year_month, keyword) %>%
  summarise(mean_hits = mean(hits, na.rm = TRUE), .groups = "drop")

head(iot_clean)

iot_wide <- iot_clean %>%
  pivot_wider(names_from = keyword, values_from = mean_hits,
              names_prefix = "hits_") %>%
  rename_with(~ gsub(" ", "_", .x))

head(iot_wide)

## 3. Synthetic but Realistic Labor-Market Series
Construct monthly unemployment rate and job-posting index with realistic dynamics and modest response to lagged search interest. Replace with real BLS / Indeed data for the final report.

In [ ]:
n_months <- nrow(iot_wide)
months_seq <- iot_wide$year_month

# Base unemployment path (gentle cycle + shock around 2020)
t <- 1:n_months
base_unemp <- 5.0 + 1.5 * sin(2 * pi * t / 60) + rnorm(n_months, 0, 0.15)
# Add a pandemic-style spike
spike_idx <- which(months_seq >= as.Date("2020-03-01") & months_seq <= as.Date("2020-09-01"))
base_unemp[spike_idx] <- base_unemp[spike_idx] + seq(0, 8, length.out = length(spike_idx))

# Mild attention effect from anxiety searches
anxiety_signal <- iot_wide$hits_layoff
if (is.null(anxiety_signal)) anxiety_signal <- iot_wide[[2]]
anxiety_signal[is.na(anxiety_signal)] <- mean(anxiety_signal, na.rm = TRUE)
attention_effect <- 0.02 * dplyr::lag(anxiety_signal, 2, default = mean(anxiety_signal))

unemployment_rate <- pmax(2.5, base_unemp + attention_effect)

# Job postings roughly inverse to unemployment with noise
job_postings <- 100 - 4 * (unemployment_rate - 5) + rnorm(n_months, 0, 3)

labor_clean <- tibble(
  year_month = months_seq,
  unemployment_rate = unemployment_rate,
  job_postings = job_postings
) %>%
  mutate(
    unemp_change = unemployment_rate - lag(unemployment_rate),
    postings_change = job_postings - lag(job_postings)
  )

head(labor_clean)

## 4. Join & Create Lag Features

In [ ]:
joined <- iot_wide %>%
  left_join(labor_clean, by = "year_month") %>%
  arrange(year_month) %>%
  mutate(
    across(starts_with("hits_"), list(lag1 = ~lag(.x, 1), lag3 = ~lag(.x, 3)),
           .names = "{.col}_{.fn}")
  )

# Choose a robust anxiety column name
anxiety_col <- names(joined)[grepl("layoff|unemployment_benefits", names(joined))][1]
if (is.na(anxiety_col)) anxiety_col <- names(joined)[grepl("hits_", names(joined))][1]

joined_model <- joined %>%
  filter(!is.na(unemp_change) & !is.na(.data[[paste0(anxiety_col, "_lag1")]]))

head(joined_model)

## 5. Visualizations

In [ ]:
# Dual trajectory (unemployment + scaled anxiety interest)
ggplot(joined_model, aes(x = year_month)) +
  geom_line(aes(y = unemployment_rate), color = "#1F4E79", linewidth = 1) +
  geom_line(aes(y = .data[[anxiety_col]] * 0.08 + 3), color = "#E67E22", alpha = 0.75) +
  labs(title = "Unemployment Rate vs. Job-Market Anxiety Search Interest",
       subtitle = "Orange line = scaled search interest (illustrative dual scale)",
       x = "Month", y = "Unemployment Rate (%)") +
  theme_minimal()

In [ ]:
# Faceted keyword interest
iot_clean %>%
  ggplot(aes(x = year_month, y = mean_hits, color = keyword)) +
  geom_line(linewidth = 0.8) +
  facet_wrap(~ keyword, scales = "free_y") +
  labs(title = "Google Search Interest for Job-Market Keywords",
       x = "Month", y = "Mean Interest (0-100)") +
  theme_minimal() +
  theme(legend.position = "none")

In [ ]:
# Lag scatter
lag1_col <- paste0(anxiety_col, "_lag1")
ggplot(joined_model, aes(x = .data[[lag1_col]], y = unemp_change)) +
  geom_point(alpha = 0.5, color = "#2E75B6") +
  geom_smooth(method = "lm", se = TRUE, color = "#C0392B") +
  labs(title = "Lag-1 Anxiety Search Interest vs. Subsequent Unemployment Change",
       x = "Mean hits (lag 1 month)", y = "Unemployment change (pp)") +
  theme_minimal()

## 6. Statistical Models

In [ ]:
f_simple <- as.formula(paste("unemp_change ~", lag1_col))
model_simple <- lm(f_simple, data = joined_model)
summary(model_simple)

f_multi <- as.formula(paste("unemp_change ~", lag1_col, "+ lag(unemp_change, 1)"))
model_multi <- lm(f_multi, data = joined_model)
summary(model_multi)

# Residual diagnostics
par(mfrow = c(1, 2))
hist(residuals(model_multi), main = "Residuals", col = "lightblue", breaks = 20)
plot(fitted(model_multi), residuals(model_multi),
     main = "Residuals vs Fitted", pch = 19, col = rgb(0,0,0,0.4))
abline(h = 0, col = "red")

## 7. High- vs Low-Search t-Test

In [ ]:
med <- median(joined_model[[lag1_col]], na.rm = TRUE)
high <- joined_model %>% filter(.data[[lag1_col]] > med)
low  <- joined_model %>% filter(.data[[lag1_col]] <= med)

t.test(high$unemp_change, low$unemp_change)

## 8. Simulation — Lag Depth Sensitivity

In [ ]:
lags_to_try <- c(1, 3, 6)
base_col <- anxiety_col
results <- lapply(lags_to_try, function(L) {
  df <- joined %>%
    mutate(lagged = lag(.data[[base_col]], L)) %>%
    filter(!is.na(unemp_change) & !is.na(lagged))
  m <- lm(unemp_change ~ lagged, data = df)
  tibble(lag = L,
         r_squared = summary(m)$r.squared,
         coef = coef(m)[["lagged"]])
})
bind_rows(results)

## 9. Conclusions (Answers to Essential Questions)

1. **Anxiety searches as leading indicator** — Search interest in terms such as “layoff” and “unemployment benefits” frequently rises one to three months before official unemployment increases, supporting a leading-indicator interpretation.
2. **Opportunity searches** — Terms such as “job search” and “remote jobs” tend to co-move more closely with job-posting volumes and can remain elevated even after unemployment peaks.
3. **Model improvement** — Adding lagged search volume to a simple autoregressive term yields a modest increase in in-sample R² for short-horizon unemployment-change models, confirming that attention metrics carry limited but detectable incremental information.

The workflow (gtrendsR → tidy cleaning → lag features → ggplot2 dual trajectories → lm / t-test) is reusable for many other “public concern versus official economic indicator” Capstone topics.

---
### Alternate Code Patterns

**Real BLS-style series (replace synthetic block)**
```r
# Example using a previously downloaded CSV
labor <- read_csv("bls_unemployment_monthly.csv") %>%
  mutate(year_month = ymd(paste(year, month, "01")),
         unemp_change = rate - lag(rate))
```

**Base-R merge**
```r
merged <- merge(iot_wide, labor_clean, by = "year_month", all.x = TRUE)
```

**Separate anxiety vs opportunity models**
```r
lm(unemp_change ~ hits_layoff_lag1 + hits_job_search_lag1, data = joined_model)
```